# Q ME NOW — Manager Performance

Computes executive manager rankings from exported branch operations data. The dashboard consumes the imported `manager_performance` insight and can also calculate the same score directly from live API data.

In [ ]:
import json, os
from datetime import datetime, timezone

import numpy as np
import pandas as pd

BASE = os.path.join(os.getcwd(), "..")
DATA = os.path.join(BASE, "data_exports")
OUTPUTS = os.path.join(BASE, "outputs")
os.makedirs(DATA, exist_ok=True)
os.makedirs(OUTPUTS, exist_ok=True)

input_path = os.path.join(DATA, "manager_performance_input.csv")
if not os.path.exists(input_path):
    raise FileNotFoundError("manager_performance_input.csv was not exported.")

df = pd.read_csv(input_path)
for col in ["total_visits", "completed_count", "no_show_count", "avg_wait_minutes", "avg_service_minutes", "assigned_staff", "counter_count"]:
    df[col] = pd.to_numeric(df.get(col, 0), errors="coerce").fillna(0)

def normalize(series, higher_is_better=True):
    s = pd.to_numeric(series, errors="coerce").fillna(0)
    hi = s.max()
    if hi <= 0:
        out = pd.Series([50] * len(s), index=s.index)
    else:
        out = (s / hi * 100).clip(0, 100)
    return out if higher_is_better else 100 - out

def util_score(row):
    counters = row["counter_count"] or 0
    util = (row["assigned_staff"] / counters) if counters else 0
    return max(0, min(100, 100 - abs(util - 0.8) * 140))

df["completion_rate"] = np.where(df["total_visits"] > 0, df["completed_count"] / df["total_visits"], 0)
df["no_show_rate"] = np.where(df["total_visits"] > 0, df["no_show_count"] / df["total_visits"], 0)
df["staff_utilization"] = np.where(df["counter_count"] > 0, df["assigned_staff"] / df["counter_count"], 0)
df["wait_score"] = normalize(df["avg_wait_minutes"], higher_is_better=False)
df["completion_score"] = (df["completion_rate"] * 100).clip(0, 100)
df["no_show_score"] = ((1 - df["no_show_rate"]) * 100).clip(0, 100)
df["throughput_score"] = normalize(df["total_visits"], higher_is_better=True)
df["staff_utilization_score"] = df.apply(util_score, axis=1)
df["manager_score"] = (
    df["wait_score"] * 0.30 +
    df["completion_score"] * 0.25 +
    df["no_show_score"] * 0.20 +
    df["throughput_score"] * 0.15 +
    df["staff_utilization_score"] * 0.10
).round(1)
df = df.sort_values("manager_score", ascending=False).reset_index(drop=True)
df["rank"] = df.index + 1
df["completion_rate"] = (df["completion_rate"] * 100).round(1)
df["no_show_rate"] = (df["no_show_rate"] * 100).round(1)
df["staff_utilization"] = (df["staff_utilization"] * 100).round(1)

summary_cols = [
    "rank", "manager_id", "manager_name", "staff_code", "business_id", "branch_id", "branch_name",
    "manager_score", "total_visits", "completed_count", "no_show_count", "completion_rate",
    "no_show_rate", "avg_wait_minutes", "avg_service_minutes", "staff_utilization"
]
out = df[summary_cols]
csv_path = os.path.join(DATA, "manager_performance_summary.csv")
out.to_csv(csv_path, index=False)

payload = {
    "generated_at": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
    "weights": {"wait_time": 0.30, "completion_rate": 0.25, "no_show_rate": 0.20, "throughput": 0.15, "staff_utilization": 0.10},
    "managers": out.to_dict("records"),
    "records_processed": int(len(out)),
}
with open(os.path.join(OUTPUTS, "manager_performance.json"), "w", encoding="utf-8") as handle:
    json.dump(payload, handle, indent=2)

print(f"manager_performance_summary.csv: {len(out)} rows")
